# Chapter 13 — 보안 Assistant 종합 구현 (Capstone)

> **Chapter 13 · Part II**

이 Part 의 기술을 하나로 통합한 capstone.

## 실습 목표
1. SecurityAssistant 클래스 통합 구현 (ML 1차 → RAG → LLM → Agent)
2. 시나리오 9 건 테스트 — 세 경로(pass·auto·llm_agent)가 모두 실제로 쓰이도록
3. 고위험 경로 확인 — sev≥4 에서 자기 검증과 승인 게이트가 실제로 실행되는지
4. 종합 평가 — 최종 판정 기준 정확도, 경로별 정확도·지연, `usage` 토큰 기반 실제 비용
5. CLI 패키징 — 저장소의 `labs/scripts/security_assistant_cli.py` 를 실제로 실행

> **주의:** API 키가 없으면 LLM 단계는 mock 으로 대체됩니다. mock 은 severity 를 규칙으로
> 정하므로 "모델이 이렇게 판단한다" 를 보여 주지 않습니다. 비용 계산은 실제 호출이 있어야
> 의미가 있습니다.

In [ ]:
# 첫 사용시에만 주석 해제하여 실행 (Colab/venv 환경에 따라 일부는 이미 설치됨)
# !pip install -q anthropic
# !pip install -q chromadb
# !pip install -q sentence-transformers
# !pip install -q pandas
# !pip install -q numpy

In [ ]:
import os, json, re, sys, time, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import chromadb
from chromadb.utils import embedding_functions

API_KEY = os.environ.get('ANTHROPIC_API_KEY')
USE_REAL = bool(API_KEY)
rng = np.random.default_rng(42)

---
## Step 1. ML 1차·RAG·LLM 컴포넌트 준비

**ML 1차는 이 노트북에서 규칙 기반 대체물**입니다. 원래 이 자리에는 전작 ch13 의
이진 RandomForest를 pickle로 불러와 씁니다. Isolation Forest는 이상점수 계약이 달라 별도 어댑터가 필요합니다. 여기서는 전작의
학습 데이터와 모델 파일 없이도 capstone 의 **라우팅 구조**를 돌려 보기 위해, 같은 인터페이스
(로그 → 0~1 점수)를 갖는 정규식·임계값 규칙으로 대신합니다. 점수 분포가 세 경로에 고르게
퍼지도록 페이로드 외에 유출량·실패 횟수 같은 수치 지표도 함께 봅니다.

RAG는 ch11의 KB를 축소해 재사용합니다. Step 5-1에서는 실제 sklearn RF 산출물을
같은 라우터에 연결하고 누락 특성의 거부를 검증합니다.

In [ ]:
# TODO: 아래에 코드를 작성하세요.
# 힌트: ML 1차 — 전작 ch13 RandomForest 자리의 규칙 기반 대체물 (같은 인터페이스: 로그 -> 0~1)

<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
# ML 1차 — 전작 ch13 RandomForest 자리의 규칙 기반 대체물 (같은 인터페이스: 로그 -> 0~1)
def ml_score(log: ...
    url, ua = ...
    if re.search(r'union\s+select|or\s+1=1|<script>|jndi: ...
        return ...
    if re.search(r'sqlmap|nikto|nuclei|masscan', ua, re.I):
        return ...
    if log.get('bytes_out', 0) >= 1_000_000_000:
        return ...
    if log.get('fail_count', 0) >= 10:
        return ...
    if log.get('status') == 401 and 'login' in url:
        return ...
    return ...


# RAG — ch11 의 KB 를 축소 재사용
ef = ...
client_chroma = ...
col = ...
if col.count() == 0:
    kb = ...
        ...
    col.add(documents = ...
            metadatas = ...

# LLM
if USE_REAL:
    import anthropic
    llm = ...

# 2026-09 공식 단가 ($/1M 토큰, 입력/출력)
PRICE_PER_MTOK = ...

print(f'ML 1차 = ...
```
</details>


---
## Step 2. SecurityAssistant 통합 클래스

본문 §13.2 와 같은 구조이되, 노트북에서 실제로 돌아가도록 두 곳을 바꿨습니다.

- **승인자**: 본문의 `_tool` 은 `input()` 으로 사람에게 묻습니다. 노트북은 비대화형이므로
  ch12 §12.5 와 같은 **정책 승인자**가 대신 판단합니다 — 사내 대역·화이트리스트 대상 차단은
  거부하고, 외부 출발지의 sev≥4 차단만 승인합니다.
- **usage 수집**: 모든 LLM 호출의 입력·출력 토큰을 모아 Step 4 에서 실제 비용을 계산합니다.
생성자의 `score_fn=None`을 `ml_score`로 기본 설정하고, analyze에서는 `self.score_fn(log)`를 호출합니다. Step 5-1에서 RF scorer를 주입할 수 있어야 합니다.


In [ ]:
# TODO: 아래에 코드를 작성하세요.
# 힌트: ch10 §10.3.1 자기 검증이 "재검토 필요"로 답하면 자동 승인하지 않는다.

<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
WHITELIST = ...
INTERNAL_PREFIX = ...


def policy_approver(name: ...
    ...
    ip = ...
    if ip in WHITELIST:
        return ...
    if ip.startswith(INTERNAL_PREFIX):
        return ...
    if not ctx.get('verified', True):
        # ch10 §10.3.1 자기 검증이 "재검토 필요"로 답하면 자동 승인하지 않는다.
        # 검증 결과가 조치를 바꾸지 않으면 그 호출은 비용만 쓰는 장식이다.
        return ...
    if ctx.get('severity', 0) >= 4:
        return ...
    return ...


class SecurityAssistant:
    def __init__(self, dry_run=True, approver=policy_approver, score_fn=None):
        self.score_fn = score_fn if score_fn is not None else ml_score
        self.dry_run = ...
        self.approver = ...
        self.audit_log = ...
        self.usage = ...

    # ---------- LLM 호출 (usage 수집 포함) ----------
    def _call(self, model: ...
        kwargs = ...
        resp = ...
                                   messages = ...
        ...
        if resp.stop_reason != 'end_turn':
            print(f'  [경고] stop_reason = ...
        return ...

    def cost_usd(self) -> float:
        total = ...
        for u in self.usage:
            pin, pout = ...
            total += ...
        return ...

    # ---------- 파이프라인 ----------
    def analyze(self, log: ...
        s = ...
        if s < 0.3:
            return ...
        if s > 0.9:
            act = ...
                             ip = ...
            return ...

        # 2) RAG
        query = ...
        hits = ...
        sources = ...
                   for d, m in zip(hits['documents'][0], hits['metadatas'][0])]

        # 3) LLM — ch10 §10.1.3 CoT 5단계를 시스템 프롬프트에 명시
        cot_steps = ...
                     ...
        if USE_REAL:
            # ch11 §11.8 — RETRIEVED 델리미터로 구획하고 지시가 아님을 명시
            ctx = ...
                            for s_ in sources)
            system = ...
                      ...
                      'severity 는 1 = ...
                      ...
            text = ...
            # 응답은 CoT 서술 + 코드펜스로 감싼 JSON 이므로 마지막 { ... } 블록만 추출한다.
            # max_tokens 은 1200 이다 — 750 에서는 CoT 서술이 길어지면 JSON 이 잘려 파싱에 실패했다.
            m = ...
            try:
                analysis = ...
            except Exception:
                analysis = ...
            if not analysis:
                analysis = ...
                            ...
        else:
            sev = ...
            analysis = ...
                        'summary': f'mock 판정 (ml = ...
                        ...

        # 4) Agent
        actions, verification = ...
        sev = ...
        if sev >= 4:
            # ch10 §10.3.1 자기 검증 — 가장 비싼 조치(차단) 직전에만 2차 호출
            verification = ...
            actions.append(self._tool('block_ip', high_risk = ...
                                      ctx = ...
                                      ip = ...
        if sev >= 3:
            actions.append(self._tool('send_slack_alert', ctx = ...
                                      channel = ...
                                      severity = ...
        if sev >= 2:
            actions.append(self._tool('create_ticket', ctx = ...
                                      title = ...
                                      description = ...
                                      priority = ...
        verdict = ...
        return ...
                ...

    def _verify_high_severity(self, log, analysis):
        ...
        if not USE_REAL:
            return ...
        text = ...
                          ...
                          f'로그: {json.dumps(log, ensure_ascii = ...
                          ...
        return ...

    def _tool(self, name, high_risk=False, ctx=None, **args):
        ...
        decision, reason = ...
        if high_risk:
            decision, reason = ...
        ...
        if decision not in ('auto', 'approve'):
            ...
            return ...
        if self.dry_run:
            ...
            return ...
        return ...
```
</details>


---
## Step 3. 시나리오 9 건

세 경로(pass·auto·llm_agent)가 모두 쓰이고, 그중 **sev≥4 로 자기 검증과 승인 게이트까지
가는 건**이 포함되도록 구성했습니다. 앞의 두 건은 정상 트래픽이고, 마지막 한 건은 ML 1 차가
즉시 차단으로 보내지만 대상이 사내 프로덕션 호스트라 **승인 게이트가 막아야 하는** 사례입니다.

In [ ]:
# TODO: 아래에 코드를 작성하세요.
# 힌트: (로그, 정답 라벨)

<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
SCENARIOS = ...
    # (로그, 정답 라벨)
    ...
    # ML 1차가 즉시 차단으로 보내지만 대상이 사내 프로덕션 호스트인 경우 — 승인 게이트가 막아야 한다
    ...

assistant = ...
RESULTS = ...
for log, truth in SCENARIOS:
    ...
    t0 = ...
    r = ...
    r['elapsed_ms'] = ...
    r['truth'] = ...
    r['log'] = ...
    ...
    line = ...
    if 'analysis' in r:
        line += ...
    ...
    if r.get('verification'):
        print(f"  [자기검증 {r['verification']['model']}] confirmed = ...
              ...
```
</details>


### Step 3-1. 고위험 경로가 실제로 실행됐는가

sev≥4 가 한 번도 나오지 않으면 자기 검증(`_verify_high_severity`)과 승인 게이트는 코드에만
있고 실행되지 않습니다. 실행 여부를 출력으로 확인합니다.

In [ ]:
high = [r for r in RESULTS if r.get('analysis', {}).get('severity', 0) >= 4]
print(f'sev>=4 건수: {len(high)}')
for r in high:
    blk = [a for a in assistant.audit_log
           if a['name'] == 'block_ip' and a['args'].get('ip') == r['log']['src'] and a['high_risk']]
    print(f"  {r['log']['src']:<15} sev={r['analysis']['severity']} "
          f"자기검증={'실행' if r['verification'] else '미실행'} "
          f"confirmed={r['verification']['confirmed'] if r['verification'] else '-'} "
          f"-> block_ip 판정={blk[-1]['decision'] if blk else '-'}")

gate = pd.DataFrame([a for a in assistant.audit_log if a['high_risk']])
print(f'\n고위험 도구 호출 {len(gate)} 건의 승인 게이트 판정:')
print(gate[['name', 'decision', 'reason']].to_string(index=False) if len(gate) else '  없음')
assert len(high) > 0, 'sev>=4 경로가 한 번도 실행되지 않았다 — 시나리오를 확인하라'
assert set(gate['decision']) >= {'approve', 'deny'}, '승인과 거부가 모두 발생해야 한다'
print('\n승인·거부가 모두 실제로 발생했습니다.')

---
## Step 4. 종합 평가 — 최종 판정 기준 정확도 · 경로별 지연 · 실제 비용

종전 평가는 `route == 'pass'` 면 BENIGN 으로 채점했습니다. 그러면 LLM 이 내놓은
severity·category 는 채점에 전혀 반영되지 않고, 사실상 ML 1 차 정규식만 채점하는
셈이었습니다. 여기서는 **최종 판정**(pass→BENIGN, auto→ATTACK, llm_agent→sev≥3 이면 ATTACK)
으로 채점하고, 경로별 정확도·지연을 따로 봅니다. 비용은 `resp.usage` 의 입력·출력 토큰에
2026-09 공식 단가를 곱한 실제 값입니다.

Step 3 의 결과를 그대로 재사용합니다 — 같은 9 건을 다시 돌리면 API 비용이 두 배가 되고
감사 로그도 중복 누적됩니다.

In [ ]:
# TODO: 아래에 코드를 작성하세요.
# 힌트: df = ...

<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
df = ...
                    ...
print(df.to_string(index = ...

acc = ...
fp = ...
fn = ...
...

...
by = ...
                             평균지연ms = ...
print(by.to_string(float_format = ...

cost = ...
n_calls = ...
tok_in = ...
tok_out = ...
...
if n_calls:
    ...
else:
    ...
```
</details>


---
## Step 5. 단건 분석 (CLI 와 같은 진입점)

본문 §13.3의 CLI(`labs/scripts/security_assistant_cli.py`)는 경량 lab의
`SecurityAssistant.analyze()`를 호출합니다. 노트북과 입력·라우팅 구조는 같지만
CLI의 RAG·LLM·대응은 모의 경로이며, `--ml-model`을 지정하면 RF만 실제 연결합니다. 저장소를 clone 한 환경에서는

```bash
python labs/scripts/security_assistant_cli.py --log '{"src":"1.2.3.4","url":"/?id=1 UNION SELECT","ua":"sqlmap/1.7"}'
```

로 실행하며, 여기서는 같은 진입점인 `assistant.analyze()` 를 두 로그에 직접 적용해 결과를
확인합니다.

In [ ]:
# CLI 와 동일한 단건 분석 — assistant.analyze(log) 를 그대로 호출
for sample in ({'src':'1.2.3.4','url':'/?id=1 UNION SELECT','ua':'sqlmap/1.7','status':200},
               {'src':'10.0.0.5','url':'/index.html','ua':'Mozilla/5.0','status':200}):
    r = assistant.analyze(sample)
    print(f"\n입력: {sample['src']} {sample['url']}")
    print(f"  route={r['route']}  verdict={r.get('verdict') or r.get('analysis',{}).get('category')}"
          f"  actions={[a.get('status') for a in r.get('actions',[])]}")

## Step 5-1. 전작 RF 산출물 연결 검증
합성 자료로 실제 전처리기·RF를 학습하고 같은 라우터에 연결합니다. 탐지 성능 측정이 아니라 컬럼·라벨 계약의 실행 검증입니다. 전작 RF는 직접 생성한 파일과 연결 통계를 담은 이벤트를 함께 지정합니다.


In [ ]:
# 외부 API·데이터 다운로드 없이 실제 sklearn 산출물 연결을 검증한다.
import pickle, tempfile, sys, os, json
from pathlib import Path
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

roots = [Path.cwd(), *Path.cwd().parents]
repo = next((p for p in roots if (p / 'labs/artifact_model.py').exists()), None)
if repo is None:
    raise FileNotFoundError('공식 실습 저장소를 내려받고 저장소 안에서 실행하세요.')
sys.path.insert(0, str(repo))
from labs.artifact_model import BinaryArtifactScorer

# 전작과 동일한 12개 입력 컬럼. 아래 값과 라벨은 연결 검증용 합성 자료다.
num = ['duration', 'src_bytes', 'dst_bytes', 'count', 'srv_count',
       'serror_rate', 'rerror_rate', 'same_srv_rate', 'diff_srv_rate']
cat = ['protocol_type', 'service', 'flag']
rows = []
for i in range(20):
    attack = i >= 10
    rows.append(dict(zip(num, [i, 100 if not attack else 9000, 300,
        1 if not attack else 80, 1, float(attack), 0, 1-float(attack), float(attack)]),
        protocol_type='tcp', service='http' if not attack else 'ssh',
        flag='SF' if not attack else 'S0'))
frame = pd.DataFrame(rows)
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat)])
model = RandomForestClassifier(n_estimators=30, random_state=42).fit(
    preprocessor.fit_transform(frame), [0]*10+[1]*10)
with tempfile.TemporaryDirectory() as folder:
    path = Path(folder) / 'synthetic_rf.pickle'
    path.write_bytes(pickle.dumps({'preprocessor': preprocessor, 'model': model,
        'feature_columns': num+cat, 'positive_label': 1}))
    scorer = BinaryArtifactScorer(path)
    event = {'src': '192.0.2.20', 'features': rows[0]}
    assert scorer(event) == model.predict_proba(preprocessor.transform(frame.iloc[[0]]))[0, 1]
    # 노트북의 같은 라우터에 실제 모델 scorer를 주입한다.
    connected = SecurityAssistant(score_fn=scorer)
    result = connected.analyze(event)
    assert result['route'] == 'pass'
    print('합성 RF 연결 검증:', result['route'], result['ml'])
    try:
        scorer({'url': '/login'})
    except ValueError:
        print('특성 없는 HTTP 로그: 추측 없이 거부됨')
    else:
        raise AssertionError('누락 특성을 거부해야 합니다.')

# 전작에서 직접 저장한 RF와 수집기가 계산한 features 이벤트를 함께 지정한다.
model_path = os.environ.get('BOOK10_RF_MODEL')
event_path = os.environ.get('BOOK10_FEATURE_EVENT')
if bool(model_path) != bool(event_path):
    raise ValueError('BOOK10_RF_MODEL과 BOOK10_FEATURE_EVENT를 함께 지정하세요.')
if model_path:
    real_scorer = BinaryArtifactScorer(model_path)
    event = json.loads(Path(event_path).read_text())
    print('전작 RF의 실제 연결 점수:', real_scorer(event))
    # 같은 라우터를 사용한다. 회색 구간은 앞 단계의 RAG·LLM 설정을 따른다.
    connected = SecurityAssistant(score_fn=real_scorer, dry_run=True)
    print(connected.analyze(event))
else:
    print('전작 RF 실행: 경로 미지정. 위 결과는 합성 산출물의 계약 검증입니다.')


---
## 정리
- ch09~ch12 에서 만든 LLM 로그 해석·Prompt Engineering·RAG·SOAR Agent 를 SecurityAssistant
  하나로 통합했습니다.
- ML 1 차 판정 → RAG 검색 → LLM 분석 → Agent 대응의 계층 라우팅으로, 모호한 케이스만 비싼
  단계로 보냅니다. 세 경로가 모두 실제로 쓰이는 시나리오로 확인했습니다.
- sev≥4 인 건에서 자기 검증(다른 모델)과 승인 게이트가 실행되고, 사내 대역 대상 차단이
  거부되는 것까지 출력으로 확인했습니다.
- 평가는 ML 점수가 아니라 **최종 판정**으로 채점하고, 경로별 정확도·지연과 `usage` 토큰
  기반 실제 비용을 분리해 봤습니다.
- CLI 는 저장소의 실제 스크립트를 실행해 확인했습니다.
- 네트워크·로그 기초와 ML 기반 탐지는 전작(『바이브 코딩으로 만드는 AI 네트워크 보안』)에서 다룹니다.

> **다음 Part:** Part III 미토스/페이블5 LLM 보안 고도화 — ch14 MCP 보안 도구 통합부터.